# Deploy Lakehouse Agent

Deploy the Lakehouse Agent to AgentCore Runtime.

## Prerequisites

- ✅ Run `05a-deploy-claims-gateway.ipynb` (GW1) **and** `05b-deploy-notes-gateway.ipynb` (GW2) first
- ✅ Gateway ARN saved to SSM
- ✅ AgentCore CLI 0.30.0 installed (`npm install -g @aws/agentcore@0.30.0`, Node.js 20+) and `uv` on PATH

## What This Notebook Does

1. Creates the agent's execution role
2. Deploys Lakehouse Agent to AgentCore Runtime with the AgentCore CLI
3. Configures Gateway integration
4. Saves Agent Runtime ARN to SSM

## Next Notebook

- **08-streamlit-ui.ipynb**

In [ ]:
# AWS Initialization - Load credentials and create session
from utils.notebook_init import init_aws

# This will:
# 1. Load credentials from .env file (if it exists)
# 2. Create and validate AWS session (env vars take precedence over SSO)
# 3. Return session, region, and account_id for use in this notebook
session, region, account_id = init_aws()

# Initialize AWS clients
ssm_client = session.client("ssm", region_name=region)

# Load Gateway ARN from SSM
try:
    gateway_arn = ssm_client.get_parameter(Name="/app/lakehouse-agent/gateway-arn")["Parameter"]["Value"]
    print("✅ Ready to proceed with AWS operations")
    print(f"   Account ID: {account_id}")
    print(f"   Region: {region}")
    print(f"   Gateway ARN: {gateway_arn}")
except ssm_client.exceptions.ParameterNotFound:
    print("✅ Ready to proceed with AWS operations")
    print(f"   Account ID: {account_id}")
    print(f"   Region: {region}")
    print("❌ Gateway ARN not found in SSM")
    print("   Please run 05a-deploy-claims-gateway.ipynb first")
    gateway_arn = None

## Step 1: Deploy Lakehouse Agent
First creates the agent's execution role, next to the runtime that uses it. Then deploys the agent's AgentCore CLI project (`deployment/6-lakehouse-agent/agentcore/`) as its own CloudFormation stack, with the claims gateway ARN from notebook 05a as `GATEWAY_ARN`.

In [ ]:
import subprocess
import sys

# Create (or update) this runtime's execution role, AgentCoreRuntimeRole-lakehouse-agent.
# Created next to the runtime that uses it, like the two MCP server roles.
result = subprocess.run(
    [sys.executable, "setup_runtime_roles.py", "create", "--role", "lakehouse-agent"],
    cwd="deployment/2-lakehouse-tenant-roles-setup",
)
if result.returncode != 0:
    print("❌ Error: setup_runtime_roles.py failed (see output above)")
else:
    print("\n✅ Execution role AgentCoreRuntimeRole-lakehouse-agent ready")

In [ ]:
import subprocess
import sys

# Deploys this runtime's AgentCore CLI project (deployment/6-lakehouse-agent/agentcore/)
# as its own CloudFormation stack, then stores the runtime ARN/ID in SSM.
# No capture_output: stdout/stderr stream live to the notebook so the deploy
# progress appears incrementally instead of all at once when the script ends.
result = subprocess.run(
    [sys.executable, "agentcore_cli_deploy.py", "lakehouse-agent"],
    cwd="deployment",
)
if result.returncode != 0:
    print("❌ Error: agentcore_cli_deploy.py failed (see output above)")
else:
    print("\n✅ Lakehouse Agent deployed!")
    print("\n📋 Runtime configuration saved to SSM Parameter Store")

## Step 2: Verify Agent Deployment

The `agentcore_cli_deploy.py` script automatically saves the Runtime ARN to SSM.
Run this cell to verify the deployment.

In [ ]:
# Verify Agent configuration in SSM
print("Verifying Agent Runtime configuration in SSM...\n")

# agent-name holds the deployed runtime name, lakehouseagent_lakehouse_agent
# (the AgentCore CLI prefixes the runtime name with its project name).
parameters_to_check = [
    "/app/lakehouse-agent/agent-runtime-arn",
    "/app/lakehouse-agent/agent-runtime-id",
    "/app/lakehouse-agent/agent-name",
]

all_found = True
for param_name in parameters_to_check:
    try:
        response = ssm_client.get_parameter(Name=param_name)
        value = response["Parameter"]["Value"]
        print(f"✅ {param_name}")
        print(f"   Value: {value}")
    except ssm_client.exceptions.ParameterNotFound:
        print(f"⚠️  {param_name} - NOT FOUND (optional)")
        if "agent-runtime-arn" in param_name:
            all_found = False
    except Exception as e:
        print(f"⚠️  {param_name} - ERROR: {e}")

if all_found:
    print("\n✅ Agent Runtime configuration verified in SSM!")
else:
    print("\n❌ Agent Runtime ARN missing in SSM.")
    print("    agentcore_cli_deploy.py should have saved this automatically.")
    print("    Check the deployment output for errors.")

## Summary

✅ **Lakehouse Agent Deployment Complete!**

The Agent Runtime has been deployed and configuration saved to SSM Parameter Store.

**Next Steps:**
Run **08-streamlit-ui.ipynb** to test the complete system
